# Generator

A Python generator is a special function that returns an iterator object, allowing you to iterate over a sequence of values one at a time on demand rather than computing and storing them all in memory at once.

### How Generators work

Instead of using the return keyword to exit a function with a final result, generators use the yield keyword. When Python encounters a yield statement:

- It pauses the function's execution.

- It saves the entire state of the function (including local variables).

- It returns the yielded value to the caller.

- It resumes exactly where it left off the next time the generator is called.

### Key benefits

- **Memory Efficiency**: They do not load massive datasets into RAM, preventing memory errors.

- **Lazy Evaluation**: Values are calculated dynamically only when you explicitly ask for them.

- **Infinite Sequences**: They can safely represent unbounded streams of data (like a Fibonacci series) without crashing your system.

In [24]:
# Defining the generator function
def gen_int(start=0, stop=0):
    while start <= stop:
        yield start                 # Pauses here and outputs the value
        start += 1

# Creating the generator object
num = gen_int(1, 5)

# Accessing values manually
print(next(num))

# do anything in between
print("Another task in between!")

print(next(num))
print(next(num))
print(next(num))
print(next(num))

# Calling next() again would raise a StopIteration error because limit/range is completed. We will see below how to resolve it seamlessly.

1
Another task in between!
2
3
4
5


---

# Code Example: Generator Expression

If you need a simple generator, you can write a **Generator Expression**. They use the exact same syntax as list comprehensions but use parentheses () instead of square brackets [].

In [1]:
# This creates the entire list in memory all at once
sqr_lst = [x**2 for x in range(1000000)]

# This creates a generator that calculates squares one by one on the fly
sqr_generator = (x**2 for x in range(1000000))

print(next(sqr_generator))
print(next(sqr_generator))

0
1


---

# Real-World Use Case: Processing Large Log Files

If you need to read a 10 GB log file, using a normal function with `.readlines()` will load the entire 10 GB file into your RAM. A generator streams the file line-by-line, utilizing only a few kilobytes of RAM.

In [ ]:
def read_large_file(file_path):
    with open(file_path, "r") as file:
        for line in file:
            yield line  # Memory-safe streaming

# Processing the stream without overloading RAM
for log_line in read_large_file("huge_system_log.txt"):
    if "ERROR" in log_line:
        print(log_line)

---

# Handling the `StopIteration` Error

When a generator finishes its loop and has no more values to `yield`, Python automatically raises a `StopIteration` exception.

If you call `next()` manually 6 times on your 1-to-5 generator, the 6th call crashes because the code ends.

In [2]:
# generator
def count_upto(maximum):
    count = 1
    while count <= maximum:
        yield count
        count += 1

# generator object
counter = count_upto(5)

# manual access
print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))
print(next(counter))

1
2
3
4
5


In [3]:
# let's now require an extra than the range we provided
print(next(counter))

StopIteration: 

> We ended-up with an exception because Python raised StopIteration as no further creation. Let's now learn how to handle such scenario.  

---

# Two Solutions

**Solution A: The default parameter (Easiest)**

You can pass a fallback value to the `next()` function. If the generator is empty, it returns that value instead of crashing.

In [5]:
# Pass a default value like "Done" or None as the second argument
print(next(counter, "Done"))  # Outputs: Done

Done


> 💥 BOOM! No error and single parameter is savior...!

**Solution B: Using a try-except block**

If you want to catch the error in a professional production script, wrap it in a standard exception handler.

In [13]:
counter_2 = count_upto(10)

while True:
    try:
        print(next(counter_2))
    except StopIteration:
        print("Generator has no more number!")
        break

1
2
3
4
5
6
7
8
9
10
Generator has no more number!


---

# More Examples

**1. Unpacking**: Instead of manually calling the generator or looping through it element by element, you can unpack its values directly into variables using standard Python unpacking syntax.

In [1]:
# generator function
def gen_nums():
    yield 1
    yield 2
    yield 3

# unpack 
a, b, c = gen_nums()

print(a, b, c)

1 2 3


---

**2. Starred Unpacking**: catches the rest into a list

In [2]:
# *rest catch all remaining values into a list
first, *rest = gen_nums()

print(first)
print(rest)

1
[2, 3]


---

**3. Unpacking into Collections**: You can instantly dump all values of a generator into collections like lists, sets, or tuples by prepending the generator expression or object with a *.

In [3]:
# Generator expression
my_gen = (x**2 for x in range(10))

# unpacking into a list
num_lst = [*my_gen]             # used * before variable/label name to get rest of values
print(num_lst)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [ ]:
# Generator expression
char_gen = (c for c in "banana")

# Unpacking into a set (removes duplicates)
char_set = {*char_gen}
print(char_set)

{'a', 'b', 'n'}


---

**4. Passing to Function Arguments**: If a function takes positional arguments, you can unpack a generator directly into the function call

In [5]:
def add_three(x, y, z):
    return x + y + z

data_gen = (var for var in [1, 2, 3])

# Unpacks into x=1, y=2, z=3
result = add_three(*data_gen)

print(result)

6


---

# ⚠️ Crucial Memory Caution

Unpacking a generator forces immediate execution of the entire generator loop.

- **The Catch**: Generators are designed to save memory by processing items one by one. Unpacking loads every single element into RAM concurrently.

- **The Risk**: Unpacking an infinite generator (like while True: yield x) or a massive generator will cause your program to freeze or crash with an OutOfMemory error.

---

# Micro-Benchmark Code

In [6]:
import timeit

# Generator yielding 10,000 integers
setup_code = """
def data_generator():
    for i in range(10_000):
        yield i
"""

# Test 1: Star Unpacking
unpack_time = timeit.timeit("[*data_generator()]", setup=setup_code, number=1000)

# Test 2: List Constructor
list_time = timeit.timeit("list(data_generator())", setup=setup_code, number=1000)

# Test 3: Manual For Loop
loop_setup = setup_code + """
def manual_loop():
    result = []
    for item in data_generator():
        result.append(item)
    return result
"""
loop_time = timeit.timeit("manual_loop()", setup=loop_setup, number=1000)

print(f"Star Unpacking:   {unpack_time:.4f} seconds")
print(f"list() Function:  {list_time:.4f} seconds")
print(f"Manual For Loop:  {loop_time:.4f} seconds")

Star Unpacking:   0.3883 seconds
list() Function:  0.3628 seconds
Manual For Loop:  0.4262 seconds
